### During coding, I noticed:
* 2 choices of search queries : free-form or structured. I first choosed structured (with city, country...) and got successfully data from "Paris" but after it was a problem with places that were not towns like "Gorges du Verdon". So I went back to structured (param "q")
* headers in the get is compulsory (or you get a weird error)
* Nominatim usage policy : limit your requests to a single thread limited to 1 machine only, no distributed scripts (including multiple Amazon EC2 instances or similar). Results must be cached on clients side. Clients sending repeatedly the same query may be classified as faulty and blocked.

In [ ]:
import requests
import pandas as pd
import time  # pour respecter la règle de Nominatim (max 1 requête/seconde)

from dotenv import load_dotenv
from pathlib import Path
import os

# --- Load .env from project root ---
project_root = Path().resolve().parent   # notebook → parent folder = repo root
load_dotenv(project_root / ".env")

url = "https://nominatim.openstreetmap.org/search"
EMAIL = os.getenv("NOMINATIM_EMAIL")

headers = {"User-Agent": f"essai-script/1.0 ({EMAIL})"}


In [ ]:
# Liste des villes et sites touristiques à chercher
best_sites_France = [
    "Mont Saint Michel","St Malo","Bayeux","Le Havre","Rouen","Paris","Amiens","Lille",
    "Strasbourg","Chateau du Haut Koenigsbourg","Colmar","Eguisheim","Besancon","Dijon",
    "Annecy","Grenoble","Lyon","Gorges du Verdon","Bormes les Mimosas","Cassis",
    "Marseille","Aix en Provence","Avignon","Uzes","Nimes","Aigues Mortes",
    "Saintes Maries de la mer","Collioure","Carcassonne","Ariege","Toulouse",
    "Montauban","Biarritz","Bayonne","La Rochelle"
]


In [ ]:
results = []

for site in best_sites_France:
    params = {
        "q": f"{site}, France", # "q" = recherche libre de Nominatim (https://nominatim.org/release-docs/latest/api/Search/) ;
        "format": "json"
    }

    r = requests.get(url, params=params, headers=headers)
    data = r.json()

    lat, lon = None, None
    if isinstance(data, list) and len(data) > 0: # verif que c'est bien une liste
        d = data[0]
        lat, lon = float(d["lat"]), float(d["lon"])

    results.append({"site": site, "lat": lat, "lon": lon})
    time.sleep(1)   # j'attends 1 seconde entre chaque requête pour respecter la règle de Nominatim (max 1 requête/seconde)

df_sites = pd.DataFrame(results)

nb_nulls = df_sites["lat"].isnull().sum()
print(f"Nombre de sites sans coordonnées trouvées : {nb_nulls}")

df_sites.to_csv("sites_lon_lat.csv", index=False)

pd.set_option("display.max_rows", None)
print(df_sites)

Nombre de sites sans coordonnées trouvées : 0
                            site        lat       lon
0              Mont Saint Michel  48.635954 -1.511460
1                        St Malo  48.649518 -2.026041
2                         Bayeux  49.276462 -0.702474
3                       Le Havre  49.493898  0.107973
4                          Rouen  49.440459  1.093966
5                          Paris  48.858890  2.320041
6                         Amiens  49.894171  2.295695
7                          Lille  50.636565  3.063528
8                     Strasbourg  48.584614  7.750713
9   Chateau du Haut Koenigsbourg  48.249411  7.344320
10                        Colmar  48.077752  7.357964
11                     Eguisheim  48.044797  7.307962
12                      Besancon  47.238022  6.024362
13                         Dijon  47.321581  5.041470
14                        Annecy  45.899235  6.128885
15                      Grenoble  45.187560  5.735782
16                          Lyon  45